# KF M1+M3 EnbPI：Oracle benchmark

這是獨立的比較實驗，不會覆蓋原本的 `KF_m1m3_EnbPI.ipynb`。Oracle 使用真正的 M1、M3 方程式與真實上一期狀態，但不知道下一期 innovation。

In [ ]:
import importlib
import pandas as pd

from kf_forecasting.models import kf_enbpi
from kf_forecasting.models import kf_oracle_benchmark

importlib.reload(kf_enbpi)
importlib.reload(kf_oracle_benchmark)

from kf_forecasting.models.kf_enbpi import EnbPIConfig, simulate_and_run, monte_carlo_summary
from kf_forecasting.models.kf_oracle_benchmark import (
    oracle_comparison_table, oracle_monte_carlo_summary,
    plot_oracle_comparison, plot_representative_oracle_run,
)
pd.set_option('display.precision', 4)
print('Loaded:', kf_enbpi.__file__)
print('Loaded:', kf_oracle_benchmark.__file__)

## 與原 M1M3 相同的 EnbPI 設定
Oracle 只提供評估基準，不會改變 bootstrap、OOB residual 或 EnbPI 區間。

In [ ]:
config = EnbPIConfig(
    window_size=15, alpha=0.05, n_bootstrap=30, block_length=None,
    batch_size=1, beta_grid_size=101, oob_bias_correction=True,
    oob_bias_correction_mode='combined',
    arima_order=None, arima_max_p=4, arima_max_q=4,
    ann_hidden_layers=(32, 16), ann_max_iter=500, ann_alpha=1e-4,
    ann_learning_rate_init=1e-3, ann_target_standardization=True,
    ann_early_stopping=False, ann_rolling_validation=True,
    ann_rolling_splits=3, ann_validation_fraction=0.10,
    ann_iteration_candidates=(125, 250, 500), ann_tol=1e-3,
    random_state=1234,
)
train_size = 650
horizon = 50

## 單次實驗：Hybrid 與 Oracle

In [ ]:
result = simulate_and_run(
    'm1m3', train_size=train_size, horizon=horizon, config=config, data_seed=2026
)
display(oracle_comparison_table(result, 'm1m3'))
plot_oracle_comparison(result, 'm1m3');

## Monte Carlo Oracle 比較
同時報告觀測 mixed data 與無 observation noise 的 clean signal RMSE。

In [ ]:
runs, enbpi_summary, mc_results = monte_carlo_summary(
    'm1m3', n_runs=20, train_size=train_size, horizon=horizon,
    config=config, seed=2026,
)
oracle_runs, oracle_summary = oracle_monte_carlo_summary(
    runs, mc_results, 'm1m3'
)
display(enbpi_summary)
display(oracle_summary)
display(oracle_runs)
plot_representative_oracle_run(runs, mc_results, 'm1m3');